In [1]:
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import torch.nn.functional as F

In [2]:
# Config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Classes
train_data = datasets.ImageFolder("images")
classes = train_data.classes
print(f"Classes: {classes}")

Using device: cpu
Classes: ['rose', 'sunflower', 'tulip']


In [3]:
# Prétraitement
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [4]:
# CNN
class NeuralNetwork(nn.Module):
    def __init__(self, classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.conv3 = nn.Conv2d(128, 256, kernel_size=3, padding=1)

        self.fc1 = nn.Linear(256 * 28 * 28, 512)
        self.fc2 = nn.Linear(512, 512)
        self.fc3 = nn.Linear(512, len(classes))

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = NeuralNetwork(classes).to(device)
print(model)


NeuralNetwork(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv3): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=200704, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=512, bias=True)
  (fc3): Linear(in_features=512, out_features=3, bias=True)
)


In [5]:
data_dir = "./images"
dataset = datasets.ImageFolder(root=data_dir, transform=transform)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [6]:
# Fonction d'entraînement
def train(dataloader, model, criterion, optimizer):
    model.train()
    running_loss = 0.0
    running_corrects = 0
    
    for inputs, labels in dataloader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        preds = torch.argmax(outputs, 1)
        running_corrects += torch.sum(preds == labels.data)

    epoch_loss = running_loss / len(dataloader.dataset)
    epoch_acc = running_corrects.double() / len(dataloader.dataset)

    print(f"Train loss: {epoch_loss:.4f}, accuracy: {epoch_acc:.4f}")

In [7]:
# Entraînement
epochs = 5
for epoch in range(epochs):
    print(f"Epoch {epoch+1}/{epochs}")
    train(dataloader, model, loss_fn, optimizer)

# Sauvegarder le modèle
torch.save(model.state_dict(), "flower_classifier_2.pth")
print("Modèle sauvegardé dans flower_classifier.pth")

Epoch 1/5
Train loss: 0.4892, accuracy: 0.7831
Epoch 2/5
Train loss: 0.3212, accuracy: 0.8675
Epoch 3/5
Train loss: 0.2877, accuracy: 0.8902
Epoch 4/5
Train loss: 0.2242, accuracy: 0.9106
Epoch 5/5
Train loss: 0.1537, accuracy: 0.9423
Modèle sauvegardé dans flower_classifier.pth


In [8]:
tensor_torch = torch.randn(1, 3, 224, 224)

# Exporter le model
torch.onnx.export(
    model,
    tensor_torch,
    "flower_classifier_2.onnx",
    input_names = ['input'],
    output_names = ['output']
)